<a href="https://www.kaggle.com/code/abidur14004/face-to-bmi?scriptVersionId=256899454" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!git clone https://github.com/liujie-zheng/face-to-bmi-vit.git
%cd face-to-bmi-vit

Cloning into 'face-to-bmi-vit'...
remote: Enumerating objects: 4224, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 4224 (delta 135), reused 181 (delta 77), pack-reused 3968 (from 1)
Receiving objects: 100% (4224/4224), 941.62 MiB | 28.82 MiB/s, done.
Resolving deltas: 100% (137/137), done.
Updating files: 100% (3979/3979), done.
/kaggle/working/face-to-bmi-vit


In [2]:
# Install required packages (check requirements from the environment files)
!pip install torch torchvision torchaudio
!pip install transformers
!pip install opencv-python
!pip install pillow
!pip install numpy pandas matplotlib
!pip install scikit-learn
# Add any other packages you see in the environment.yml files


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 782.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [3]:
import numpy as np
import cv2
import pandas as pd
from pathlib import Path
import pickle
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt

class NumpyImageDataset:
    """
    Memory-efficient NumPy-based image dataset for Kaggle environments
    Supports batch loading, preprocessing, and data augmentation
    """
    
    def __init__(self, images=None, labels=None, img_size=(224, 224), normalize=True):
        self.images = images
        self.labels = labels
        self.img_size = img_size
        self.normalize = normalize
        self.current_idx = 0
        
    def load_from_directory(self, data_dir, csv_file=None, img_col='image_path', label_col='bmi'):
        """
        Load images from directory structure
        Args:
            data_dir: Path to image directory
            csv_file: Path to CSV with image paths and labels
            img_col: Column name for image paths
            label_col: Column name for BMI labels
        """
        print("Loading images from directory...")
        
        if csv_file:
            df = pd.read_csv(csv_file)
            image_paths = [os.path.join(data_dir, path) for path in df[img_col]]
            labels = df[label_col].values
        else:
            # Auto-discover images
            image_paths = list(Path(data_dir).glob('**/*.jpg')) + \
                         list(Path(data_dir).glob('**/*.png')) + \
                         list(Path(data_dir).glob('**/*.jpeg'))
            labels = None  # No labels available
            
        images = []
        valid_labels = []
        
        for i, img_path in enumerate(tqdm(image_paths, desc="Loading images")):
            try:
                img = cv2.imread(str(img_path))
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, self.img_size)
                    images.append(img)
                    
                    if labels is not None:
                        valid_labels.append(labels[i])
                        
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
                continue
        
        self.images = np.array(images, dtype=np.uint8)
        self.labels = np.array(valid_labels) if valid_labels else None
        
        print(f"Loaded {len(self.images)} images with shape {self.images.shape}")
        return self
    
    def preprocess_images(self):
        """Preprocess images (normalize, convert to float)"""
        if self.images is None:
            raise ValueError("No images loaded")
            
        print("Preprocessing images...")
        
        # Convert to float32 and normalize
        self.images = self.images.astype(np.float32)
        
        if self.normalize:
            self.images = self.images / 255.0
            # Optional: Apply ImageNet normalization
            # mean = np.array([0.485, 0.456, 0.406])
            # std = np.array([0.229, 0.224, 0.225])
            # self.images = (self.images - mean) / std
            
        return self
    
    def augment_data(self, augment_factor=2):
        """
        Simple data augmentation
        Args:
            augment_factor: How many times to augment the data
        """
        if self.images is None:
            raise ValueError("No images loaded")
            
        print(f"Augmenting data by factor of {augment_factor}...")
        
        augmented_images = [self.images]
        augmented_labels = [self.labels] if self.labels is not None else [None]
        
        for _ in range(augment_factor - 1):
            aug_batch = []
            for img in self.images:
                # Random horizontal flip
                if np.random.random() > 0.5:
                    img = np.fliplr(img)
                
                # Random brightness adjustment
                brightness = np.random.uniform(0.8, 1.2)
                img = np.clip(img * brightness, 0, 1 if self.normalize else 255)
                
                # Random rotation (small angle)
                angle = np.random.uniform(-10, 10)
                center = (img.shape[1]//2, img.shape[0]//2)
                M = cv2.getRotationMatrix2D(center, angle, 1.0)
                img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
                
                aug_batch.append(img)
                
            augmented_images.append(np.array(aug_batch))
            if self.labels is not None:
                augmented_labels.append(self.labels.copy())
        
        self.images = np.concatenate(augmented_images, axis=0)
        if self.labels is not None:
            self.labels = np.concatenate(augmented_labels, axis=0)
            
        print(f"Dataset size after augmentation: {len(self.images)}")
        return self
    
    def train_test_split(self, test_size=0.2, random_state=42):
        """Split dataset into train and test sets"""
        if self.images is None:
            raise ValueError("No images loaded")
            
        if self.labels is not None:
            X_train, X_test, y_train, y_test = train_test_split(
                self.images, self.labels, test_size=test_size, random_state=random_state
            )
            
            train_dataset = NumpyImageDataset(X_train, y_train, self.img_size, self.normalize)
            test_dataset = NumpyImageDataset(X_test, y_test, self.img_size, self.normalize)
            
            return train_dataset, test_dataset
        else:
            # Split without labels
            split_idx = int(len(self.images) * (1 - test_size))
            indices = np.random.permutation(len(self.images))
            
            train_idx = indices[:split_idx]
            test_idx = indices[split_idx:]
            
            train_dataset = NumpyImageDataset(self.images[train_idx], None, self.img_size, self.normalize)
            test_dataset = NumpyImageDataset(self.images[test_idx], None, self.img_size, self.normalize)
            
            return train_dataset, test_dataset
    
    def get_batch(self, batch_size=32, shuffle=True):
        """
        Generator for batching data
        Args:
            batch_size: Size of each batch
            shuffle: Whether to shuffle data
        """
        if self.images is None:
            raise ValueError("No images loaded")
            
        indices = np.arange(len(self.images))
        if shuffle:
            np.random.shuffle(indices)
            
        for i in range(0, len(indices), batch_size):
            batch_indices = indices[i:i + batch_size]
            batch_images = self.images[batch_indices]
            batch_labels = self.labels[batch_indices] if self.labels is not None else None
            
            yield batch_images, batch_labels
    
    def save_dataset(self, filepath):
        """Save dataset to disk"""
        data = {
            'images': self.images,
            'labels': self.labels,
            'img_size': self.img_size,
            'normalize': self.normalize
        }
        
        with open(filepath, 'wb') as f:
            pickle.dump(data, f)
        print(f"Dataset saved to {filepath}")
    
    @classmethod
    def load_dataset(cls, filepath):
        """Load dataset from disk"""
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            
        dataset = cls(
            images=data['images'],
            labels=data['labels'],
            img_size=data['img_size'],
            normalize=data['normalize']
        )
        print(f"Dataset loaded from {filepath}")
        return dataset
    
    def get_stats(self):
        """Get dataset statistics"""
        if self.images is None:
            return "No images loaded"
            
        stats = {
            'num_images': len(self.images),
            'image_shape': self.images.shape[1:],
            'memory_usage_mb': self.images.nbytes / (1024 * 1024),
            'dtype': self.images.dtype
        }
        
        if self.labels is not None:
            stats.update({
                'num_labels': len(self.labels),
                'label_range': (self.labels.min(), self.labels.max()),
                'label_mean': self.labels.mean(),
                'label_std': self.labels.std()
            })
            
        return stats
    
    def visualize_samples(self, n_samples=9, figsize=(12, 8)):
        """Visualize sample images"""
        if self.images is None:
            raise ValueError("No images loaded")
            
        fig, axes = plt.subplots(3, 3, figsize=figsize)
        axes = axes.ravel()
        
        indices = np.random.choice(len(self.images), min(n_samples, len(self.images)), replace=False)
        
        for i, idx in enumerate(indices):
            img = self.images[idx]
            
            # Handle normalization for display
            if self.normalize and img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
                
            axes[i].imshow(img)
            axes[i].axis('off')
            
            if self.labels is not None:
                axes[i].set_title(f'BMI: {self.labels[idx]:.2f}')
                
        plt.tight_layout()
        plt.show()
    
    def __len__(self):
        return len(self.images) if self.images is not None else 0
    
    def __getitem__(self, idx):
        if self.images is None:
            raise ValueError("No images loaded")
            
        if self.labels is not None:
            return self.images[idx], self.labels[idx]
        else:
            return self.images[idx]

# Example usage functions
def create_face_bmi_dataset(data_dir, csv_file=None, img_size=(224, 224), augment=False):
    """
    Helper function to create a face-to-BMI dataset
    
    Args:
        data_dir: Directory containing images
        csv_file: CSV file with image paths and BMI values
        img_size: Target image size
        augment: Whether to apply data augmentation
    """
    dataset = NumpyImageDataset(img_size=img_size)
    
    # Load images
    dataset.load_from_directory(data_dir, csv_file)
    
    # Preprocess
    dataset.preprocess_images()
    
    # Augment if requested
    if augment:
        dataset.augment_data(augment_factor=3)
    
    return dataset

def prepare_kaggle_dataset(input_path, output_path=None, test_split=0.2):
    """
    Prepare dataset for Kaggle environment
    
    Args:
        input_path: Path to input data (CSV or directory)
        output_path: Path to save processed dataset
        test_split: Fraction for test set
    """
    print("Preparing Kaggle dataset...")
    
    # Determine if input is CSV or directory
    if input_path.endswith('.csv'):
        data_dir = os.path.dirname(input_path)
        csv_file = input_path
    else:
        data_dir = input_path
        csv_file = None
    
    # Create dataset
    full_dataset = create_face_bmi_dataset(data_dir, csv_file, augment=True)
    
    # Split dataset
    train_dataset, test_dataset = full_dataset.train_test_split(test_size=test_split)
    
    # Print stats
    print("Training set stats:")
    print(train_dataset.get_stats())
    print("\nTest set stats:")
    print(test_dataset.get_stats())
    
    # Save if output path provided
    if output_path:
        train_path = output_path.replace('.pkl', '_train.pkl')
        test_path = output_path.replace('.pkl', '_test.pkl')
        
        train_dataset.save_dataset(train_path)
        test_dataset.save_dataset(test_path)
    
    return train_dataset, test_dataset

# Example usage in Kaggle notebook:
"""
# In Kaggle notebook:

# Method 1: Load from directory with CSV
train_dataset, test_dataset = prepare_kaggle_dataset(
    input_path='/kaggle/input/face-bmi-dataset/labels.csv',
    output_path='/kaggle/working/face_bmi_dataset.pkl'
)

# Method 2: Load existing processed dataset
# train_dataset = NumpyImageDataset.load_dataset('/kaggle/input/processed-data/train_dataset.pkl')
# test_dataset = NumpyImageDataset.load_dataset('/kaggle/input/processed-data/test_dataset.pkl')

# Visualize samples
train_dataset.visualize_samples()

# Use in training loop
for epoch in range(num_epochs):
    for batch_images, batch_labels in train_dataset.get_batch(batch_size=32):
        # Your training code here
        # batch_images shape: (batch_size, height, width, channels)
        # batch_labels shape: (batch_size,) - BMI values
        pass
"""

"\n# In Kaggle notebook:\n\n# Method 1: Load from directory with CSV\ntrain_dataset, test_dataset = prepare_kaggle_dataset(\n    input_path='/kaggle/input/face-bmi-dataset/labels.csv',\n    output_path='/kaggle/working/face_bmi_dataset.pkl'\n)\n\n# Method 2: Load existing processed dataset\n# train_dataset = NumpyImageDataset.load_dataset('/kaggle/input/processed-data/train_dataset.pkl')\n# test_dataset = NumpyImageDataset.load_dataset('/kaggle/input/processed-data/test_dataset.pkl')\n\n# Visualize samples\ntrain_dataset.visualize_samples()\n\n# Use in training loop\nfor epoch in range(num_epochs):\n    for batch_images, batch_labels in train_dataset.get_batch(batch_size=32):\n        # Your training code here\n        # batch_images shape: (batch_size, height, width, channels)\n        # batch_labels shape: (batch_size,) - BMI values\n        pass\n"